In [1]:
import os
import numpy as np
import pandas as pd
from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

engine = create_engine(URL.create(
    "postgresql+psycopg2", username="ledger_user",
    password=getpass("Postgres password: "),
    host="localhost", port=5432, database="retail"))

OUT = "../dashboard/data"
os.makedirs(OUT, exist_ok=True)
print(pd.read_sql("SELECT COUNT(*) AS rows FROM transactions", engine))   # 1033068

      rows
0  1033068


In [3]:
monthly = pd.read_sql("""
    SELECT DATE_TRUNC('month', invoice_date)::date AS month,
           ROUND(SUM(revenue)) AS net_revenue,
           COUNT(DISTINCT invoice) FILTER (WHERE NOT is_cancellation) AS invoices,
           COUNT(DISTINCT customer_id) AS customers
    FROM transactions GROUP BY 1 ORDER BY 1
""", engine)
monthly["net_revenue"] = monthly["net_revenue"].astype(float)
monthly["partial_month"] = monthly["month"].astype(str).eq("2011-12-01")
monthly.to_csv(f"{OUT}/monthly_revenue.csv", index=False, encoding="utf-8")

cohort = pd.read_sql("SELECT cohort_month, month_index, active_customers FROM cohort_counts", engine)
cohort["month_index"] = cohort["month_index"].astype(int)
size = cohort[cohort["month_index"] == 0].set_index("cohort_month")["active_customers"]
cohort["cohort_size"] = cohort["cohort_month"].map(size)
cohort["retention_pct"] = (100 * cohort["active_customers"] / cohort["cohort_size"]).round(1)

cm = pd.to_datetime(cohort["cohort_month"])
month_number = cm.dt.year * 12 + cm.dt.month + cohort["month_index"]   # calendar month of each cell
cohort["partial_period"] = month_number.eq(2011 * 12 + 12)             # December 2011 is incomplete
cohort["inflated_cohort"] = cm.eq(pd.Timestamp("2009-12-01"))          # existing customers counted as new
cohort.to_csv(f"{OUT}/cohort_retention.csv", index=False, encoding="utf-8")

print(monthly.shape, cohort.shape)       # expect (25, 5) and (325, 7)
print(cohort["partial_period"].sum(), "partial-period rows")     # expect 25

(25, 5) (325, 7)
25 partial-period rows


In [4]:
seg = pd.read_sql("SELECT * FROM rfm_segments", engine)
seg["monetary"] = seg["monetary"].astype(float)
seg["decile"] = pd.qcut(seg["monetary"].rank(method="first", ascending=False),
                        10, labels=False) + 1
seg.to_csv(f"{OUT}/customers_segments.csv", index=False, encoding="utf-8")

actions = {
    "Champions":         (2, "Protect",   "Early access and loyalty perks; watch for drops in activity"),
    "Cannot lose":       (1, "Highest",   "Personal outreach or win-back offer"),
    "At risk":           (3, "High",      "Reminder plus a modest offer"),
    "Loyal":             (4, "Maintain",  "Cross-sell; move towards Champions"),
    "New and promising": (5, "Nurture",   "Second-purchase incentive"),
    "Needs attention":   (6, "Low-medium","Light-touch reminder"),
    "Lapsed":            (7, "Minimal",   "Low-cost email only"),
}
summary = (seg.groupby("segment")
           .agg(customers=("customer_id", "size"), revenue=("monetary", "sum"),
                avg_recency_days=("recency_days", "mean"), avg_frequency=("frequency", "mean"))
           .reset_index())
summary["pct_customers"] = (100 * summary["customers"] / summary["customers"].sum()).round(1)
summary["pct_revenue"] = (100 * summary["revenue"] / summary["revenue"].sum()).round(1)
summary["revenue_per_customer"] = (summary["revenue"] / summary["customers"]).round(0)
summary["sort_order"] = summary["segment"].map(lambda s: actions[s][0])
summary["priority"] = summary["segment"].map(lambda s: actions[s][1])
summary["suggested_action"] = summary["segment"].map(lambda s: actions[s][2])
summary = summary.sort_values("sort_order").round(1)
summary.to_csv(f"{OUT}/segment_summary.csv", index=False, encoding="utf-8")

conc = (seg.groupby("decile").agg(customers=("customer_id", "size"), revenue=("monetary", "sum"))
        .reset_index())
conc["pct_revenue"] = (100 * conc["revenue"] / conc["revenue"].sum()).round(1)
conc["cumulative_pct"] = conc["pct_revenue"].cumsum().round(1)
conc.to_csv(f"{OUT}/revenue_concentration.csv", index=False, encoding="utf-8")

print(summary[["segment", "customers", "pct_revenue", "priority"]].to_string(index=False))
print(conc.to_string(index=False))

          segment  customers  pct_revenue   priority
      Cannot lose        289          5.6    Highest
        Champions       1395         69.7    Protect
          At risk        424          2.8       High
            Loyal       1181         14.8   Maintain
New and promising        509          1.7    Nurture
  Needs attention        414          1.4 Low-medium
           Lapsed       1620          4.1    Minimal
 decile  customers     revenue  pct_revenue  cumulative_pct
      1        584 10360214.51         63.1            63.1
      2        583  2220359.90         13.5            76.6
      3        583  1303237.26          7.9            84.5
      4        583   850034.76          5.2            89.7
      5        583   589712.63          3.6            93.3
      6        583   418946.12          2.6            95.9
      7        583   287223.51          1.7            97.6
      8        583   197201.15          1.2            98.8
      9        583   126090.93      

In [5]:
import shutil
for f in ["backtest_summary", "backtest_sensitivity", "backtest_controls", "backtest_cost_sensitivity"]:
    shutil.copy(f"../results/{f}.csv", f"{OUT}/{f}.csv")
print(sorted(os.listdir(OUT)))

['backtest_controls.csv', 'backtest_cost_sensitivity.csv', 'backtest_sensitivity.csv', 'backtest_summary.csv', 'cohort_retention.csv', 'customers_segments.csv', 'monthly_revenue.csv', 'revenue_concentration.csv', 'segment_summary.csv']
